In [46]:
!pip install pyarabic
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import random

import scipy.sparse
import re
import string
import pyarabic.araby as araby
import nltk
from nltk.corpus import stopwords
import textblob
from textblob import Word
from sklearn import preprocessing

from wordcloud import WordCloud
import camel_tools
import warnings
from camel_tools.utils.dediac import dediac_safebw
from camel_tools.utils.charmap import CharMapper
from camel_tools.utils.transliterate import Transliterator
from sklearn.feature_extraction.text import TfidfVectorizer
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar,
    normalize_alef_maksura_ar
)

In [11]:
from datasets import load_dataset

ds = load_dataset("amgadhasan/arabic_tweets_dialects")

In [12]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'dialect'],
        num_rows: 147725
    })
})

In [13]:
df = ds["train"].to_pandas()

In [14]:
df.head()

,text,dialect
0,@toha_Altomy @gy_yah قليلين ادب ومنافقين. لو ا...,LY
1,@AlmFaisal 😂😂 الليبيين متقلبين!!!\nبس بالنسبة ...,LY
2,@smsm071990 @ALMOGRBE كل 20 تانيه شاب ليبي بير...,LY
3,@AboryPro @lyranoo85 رانيا عقليتك متخلفة. اولا...,LY
4,@lyranoo85 شكلك متعقدة علشان الراجل لي تحبيه ا...,LY


In [15]:
df.size,df.shape

(295450, (147725, 2))

In [16]:
df.isna().sum()

,0
text,0
dialect,0


In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
df["dialect"].value_counts(normalize=True) * 100

,proportion
dialect,
EG,39.015739
LY,24.707395
LB,18.694872
SD,9.770858
MA,7.811136


we need to apply class weights to avoid the imbalance

## **Define functions for preprocessing**

In [30]:
punctuations_list = string.punctuation + "،؛؟«»ـ"


def remove_punctuations(text):
    translator = str.maketrans('', '', punctuations_list)
    return text.translate(translator)

In [32]:
def remove_non_arabic(text):
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[A-Za-z0-9]", " ", text)
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    return text

In [33]:
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\U00002700-\U000027BF"
        "\U00002600-\U000026FF"
        "]+",
        flags=re.UNICODE
    )

    return emoji_pattern.sub("", text)

In [34]:
def remove_diacritics(text):
    return dediac_ar(text)


def normalize_arabic(text):
    text = normalize_alef_ar(text)
    text = normalize_alef_maksura_ar(text)
    return text


def remove_repeating_characters(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

## **Text Preprocessing pipeline**

In [38]:
nltk.download("stopwords")
stop=stopwords.words('arabic')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [39]:
stop

['إذ',
 'إذا',
 'إذما',
 'إذن',
 'أف',
 'أقل',
 'أكثر',
 'ألا',
 'إلا',
 'التي',
 'الذي',
 'الذين',
 'اللاتي',
 'اللائي',
 'اللتان',
 'اللتيا',
 'اللتين',
 'اللذان',
 'اللذين',
 'اللواتي',
 'إلى',
 'إليك',
 'إليكم',
 'إليكما',
 'إليكن',
 'أم',
 'أما',
 'أما',
 'إما',
 'أن',
 'إن',
 'إنا',
 'أنا',
 'أنت',
 'أنتم',
 'أنتما',
 'أنتن',
 'إنما',
 'إنه',
 'أنى',
 'أنى',
 'آه',
 'آها',
 'أو',
 'أولاء',
 'أولئك',
 'أوه',
 'آي',
 'أي',
 'أيها',
 'إي',
 'أين',
 'أين',
 'أينما',
 'إيه',
 'بخ',
 'بس',
 'بعد',
 'بعض',
 'بك',
 'بكم',
 'بكم',
 'بكما',
 'بكن',
 'بل',
 'بلى',
 'بما',
 'بماذا',
 'بمن',
 'بنا',
 'به',
 'بها',
 'بهم',
 'بهما',
 'بهن',
 'بي',
 'بين',
 'بيد',
 'تلك',
 'تلكم',
 'تلكما',
 'ته',
 'تي',
 'تين',
 'تينك',
 'ثم',
 'ثمة',
 'حاشا',
 'حبذا',
 'حتى',
 'حيث',
 'حيثما',
 'حين',
 'خلا',
 'دون',
 'ذا',
 'ذات',
 'ذاك',
 'ذان',
 'ذانك',
 'ذلك',
 'ذلكم',
 'ذلكما',
 'ذلكن',
 'ذه',
 'ذو',
 'ذوا',
 'ذواتا',
 'ذواتي',
 'ذي',
 'ذين',
 'ذينك',
 'ريث',
 'سوف',
 'سوى',
 'شتان',
 'عدا',
 'عسى',
 'عل'

In [48]:

def preprocessing(data: pd.DataFrame) -> pd.DataFrame:
  data["text"] = data["text"].apply(remove_punctuations)
  data["text"] = data["text"].apply(remove_emojis)
  data["text"] = data["text"].apply(remove_non_arabic)
  data["text"] = data["text"].apply(remove_repeating_characters)
  data["text"] = data["text"].apply(normalize_arabic)
  data["text"] = data["text"].apply(remove_diacritics)

  return data

In [49]:
data = preprocessing(df)

In [53]:
data.sample(5)

,text,dialect
142352,لو المرة دقو ليها ولدها بتجي كارة توبها في الو...,SD
52152,والله فعلا الخلايجة ما هم الا رقم بيتحط مفيش...,EG
106234,متعود علي البهدله الشتيمه ماعاد تفرق معو شي,LB
51544,اظن اننا محتاجين نسمع تاني سائق التوك توك,EG
56899,انا تقريبا في جيبي ورقات بفئات مختلفة من الفل...,EG


In [54]:
## Label Encoding

data.dialect.unique()

array(['LY', 'MA', 'EG', 'LB', 'SD'], dtype=object)

In [55]:
# Create a dictionary to map the labels to their encoded values
label_map = {'EG': 0, 'LY': 2, 'LB': 1, 'SD': 4, 'MA': 3}

# Create a new column with the encoded labels
data['dialect']=data['dialect'].map(label_map)

In [57]:
data.sample(5)

,text,dialect
106196,لان القيمين علي تطبيق الشرع ما عم يخافوا الل...,1
77848,لسه طالع من ع القهوة و كانت هي القاسم المشتر...,0
142992,سمر انتي بتتكلمي عن شنو,4
85390,انها سيناء اللي طلعان عين الناس اللي عايشين في...,0
88009,بس هما مصرين يخسرونا مهما كانوا حلوبين,0


In [59]:
print(data.shape)

(147725, 2)


In [60]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147725 entries, 0 to 147724
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   text     147725 non-null  object
 1   dialect  147725 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.3+ MB


In [61]:
data['dialect'].value_counts()

,count
dialect,
0,57636
2,36499
1,27617
4,14434
3,11539
